# Distilled Tiny RNNs — rarity at 100 trials

This is the next prospectively scheduled part of the frozen main study. It runs **5%, 10% and 25% rare people**, both rare strategies, and three populations per cell: **18 fresh populations**. Every population has **1,000 people and 100 trials**. The architecture, 4,000-step student budget, selection rules, classical controls, diagnostic metrics and behavioral assay follow the existing design.

Keep the complete **`runs/main-study-first-tranche-v1`** folder in Google Drive. Its verified 2% results provide a read-only baseline. See [the rarity run guide](https://github.com/WeaversMask/distilled-tiny-rnns-benchmark/blob/main/docs/rarity-tranche.md) for the exact verified 2% baseline bundle. The new run validates that baseline and saves a portable reference containing its summary, protocol and hash binding. Original baseline checkpoints and predictions stay in their original folder.

Open this notebook in a fresh Colab session and select a **T4 GPU**. The first cell downloads a fixed public revision without a GitHub token and mounts your Google Drive for results. Run the five code cells in order. Results save under **`runs/main-study-rarity-v1`**. Use one runtime per output folder.

Each invocation works on at most one pending rarity cell, with the existing soft 30-minute invocation and 10-minute phase limits. **Rerun only code cell 4 after a pause or cell completion while more cells remain.** After a deleted runtime, restore cells 1–3 before resuming cell 4. Keep the same run name. Final tables appear after all 18 populations finish. Completed cells retain their checkpoints and results.

For scheduling, the prior six-population 100-trial run took 118.86 minutes of measured T4 runner time. Three times that is roughly 6 hours for 18 populations, before additional setup and idle time. This is a rough scheduling reference; actual timing and eligible cost proxies are recorded by this run. The completed first-tranche findings remain visible, including their improved primary assay and worsened diagnostic errors.


In [ ]:
"""Download one pinned public GitHub revision for a Colab launcher.

This standard-library module is embedded into the launchers. Each invocation
installs into a fresh runtime-local destination; saved runs belong elsewhere.
"""
from pathlib import Path, PurePosixPath
import io
import os
import re
import shutil
import stat
import tempfile
import urllib.error
import urllib.parse
import urllib.request
import zipfile


_ARCHIVE_LIMIT = 50 * 1024 * 1024
_EXTRACTED_LIMIT = 100 * 1024 * 1024


class _GitHubRedirectHandler(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        parsed = urllib.parse.urlsplit(newurl)
        if (parsed.scheme != "https" or parsed.hostname != "codeload.github.com"
                or parsed.username is not None or parsed.password is not None
                or parsed.port not in (None, 443)):
            raise RuntimeError("GitHub returned an unexpected download destination")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def _extract_project_archive(payload, destination):
    """Validate all archive paths before extracting a single GitHub root."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if os.path.lexists(destination):
        raise ValueError("Use a fresh empty destination path for the project")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        if not entries or len(entries) > 10000:
            raise ValueError("Invalid project archive entry count")
        roots, seen, planned = set(), set(), []
        total_size = 0
        for entry in entries:
            name = entry.filename
            components = name.rstrip("/").split("/")
            mode = stat.S_IFMT(entry.external_attr >> 16)
            if (name.startswith("/") or "\\" in name or any(p in ("", ".", "..") for p in components)
                    or mode not in (0, stat.S_IFREG, stat.S_IFDIR) or entry.flag_bits & 1):
                raise ValueError("Unsafe project archive entry")
            roots.add(components[0])
            relative = PurePosixPath(*components[1:])
            normalized = str(relative)
            if normalized in seen:
                raise ValueError("Duplicate project archive entry")
            seen.add(normalized)
            if len(components) == 1:
                if not entry.is_dir():
                    raise ValueError("Project archive must contain a root directory")
                continue
            total_size += entry.file_size
            if total_size > _EXTRACTED_LIMIT:
                raise ValueError("Project archive is too large")
            planned.append((entry, relative))
        if len(roots) != 1 or not planned:
            raise ValueError("Project archive must have one root directory")
        files = {str(path) for entry, path in planned if not entry.is_dir()}
        for _, path in planned:
            if any(str(parent) in files for parent in path.parents):
                raise ValueError("Conflicting project archive paths")
        staging = Path(tempfile.mkdtemp(prefix=".colab-project-", dir=destination.parent))
        try:
            for entry, relative in planned:
                target = staging.joinpath(*relative.parts)
                if entry.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(entry) as source, target.open("xb") as output:
                        shutil.copyfileobj(source, output, length=1024 * 1024)
            # Caller selects an unused path. Recheck to avoid replacing a path
            # that appeared while downloading/extracting.
            if os.path.lexists(destination):
                raise ValueError("Project destination already exists")
            staging.rename(destination)
        finally:
            if staging.exists():
                shutil.rmtree(staging)
    return destination


def fetch_public_project(repo, commit, destination):
    """Fetch and safely extract a public ZIP pinned to a full Git commit."""
    if (not isinstance(repo, str)
            or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9-]{0,38}/[A-Za-z0-9_.-]{1,100}", repo)
            or repo.rsplit("/", 1)[-1] in {".", ".."}):
        raise ValueError("Repository must be an owner/name pair")
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", commit):
        raise ValueError("A full 40-character Git commit is required")
    if os.path.lexists(destination):
        raise ValueError("Use a fresh destination path for the project")
    request = urllib.request.Request(
        f"https://github.com/{repo}/archive/{commit}.zip",
        headers={"User-Agent": "distilled-rnns-colab"},
    )
    try:
        opener = urllib.request.build_opener(_GitHubRedirectHandler())
        with opener.open(request, timeout=60) as response:
            payload = response.read(_ARCHIVE_LIMIT + 1)
    except urllib.error.HTTPError as error:
        if error.code == 404:
            raise RuntimeError(
                "GitHub could not find the pinned revision. Check that the repository is public."
            ) from None
        raise RuntimeError(f"GitHub project download failed (HTTP {error.code}).") from None
    except (OSError, RuntimeError, ValueError):
        raise RuntimeError("GitHub project download failed. Check your connection and try again.") from None
    if len(payload) > _ARCHIVE_LIMIT:
        raise RuntimeError("The project download exceeds the size limit")
    try:
        return _extract_project_archive(payload, destination)
    except Exception:
        raise RuntimeError("The downloaded project archive could not be safely installed") from None

from pathlib import Path
import os, sys, hashlib, json, subprocess, tempfile
EXPECTED_SOURCE_SHA256 = "a4582314a2b34289883f526366298f6604bb4c1e327856d068ee91e2c494c9df"
EXPECTED_PROTOCOL_SHA256 = "1e8762df6fc01919dd3cfc432e96b78ed1d3fe1a25c26859edd9674cd501821e"
GITHUB_REPOSITORY = "WeaversMask/distilled-tiny-rnns-benchmark"
PINNED_COMMIT = "221e14dfc4aa73ec71799c680ae99b43bfea4fba"
RARITY_REVISION_READY = True
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not RARITY_REVISION_READY:
    raise RuntimeError("This rarity-study launcher is awaiting its verified runtime release.")
if IN_COLAB:
    destination = Path(tempfile.mkdtemp(prefix="distilled-main-source-")) / "project"
    PROJECT = fetch_public_project(GITHUB_REPOSITORY, PINNED_COMMIT, destination)
else:
    PROJECT = Path(os.environ.get("DISTILLED_PROJECT_ROOT", Path.cwd())).resolve()
    if not (PROJECT / "pyproject.toml").exists() and (PROJECT.parent / "pyproject.toml").exists():
        PROJECT = PROJECT.parent
source_paths = []
for folder in ("main_study", "foundation", "src/distilled_rnns", "grid_study"):
    paths = list((PROJECT / folder).glob("*.py"))
    if not paths:
        raise RuntimeError("Downloaded source differs from the verified notebook revision.")
    source_paths.extend(paths)
source_paths.extend([PROJECT / "configs/main_study_v1.json", PROJECT / "configs/grid_baseline_reference_v1.json"])
digest = hashlib.sha256()
for path in sorted(source_paths, key=lambda value: value.relative_to(PROJECT).as_posix()):
    digest.update(path.relative_to(PROJECT).as_posix().encode())
    digest.update(path.read_bytes())
if digest.hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("Downloaded source differs from the verified notebook revision.")
if hashlib.sha256((PROJECT / "configs/main_study_v1.json").read_bytes()).hexdigest() != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError("Downloaded prospective protocol differs from the verified revision.")
if IN_COLAB:
    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/distilled-tiny-rnns/runs")
else:
    RUNS = Path(os.environ.get("DISTILLED_OUTPUT_ROOT", str(PROJECT / "outputs/notebook-main-study")))
os.chdir(PROJECT)
RUNS.mkdir(parents=True, exist_ok=True)
print("Verified project:", PROJECT)
print("Results:", RUNS)


## Install and verify

Verify the pinned runtime and unchanged frozen protocol, activate source, and run the full preflight suite on the selected device.

In [ ]:
# Reject a previously imported revision before activating this source.
for namespace, relative in (("distilled_rnns", "src/distilled_rnns"), ("main_study", "main_study"), ("foundation", "foundation"), ("grid_study", "grid_study")):
    expected = (PROJECT / relative).resolve()
    for name, module in list(sys.modules.items()):
        if name == namespace or name.startswith(namespace + "."):
            module_file = getattr(module, "__file__", None)
            locations = [Path(module_file)] if module_file else [Path(p) for p in getattr(module, "__path__", [])]
            if not locations or any(not p.resolve().is_relative_to(expected) for p in locations):
                raise RuntimeError("Another project revision is loaded. Restart the runtime and run from cell one.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[test]"], check=True)
for path in (PROJECT, PROJECT / "src"):
    if str(path.resolve()) not in sys.path:
        sys.path.insert(0, str(path.resolve()))
import importlib
importlib.invalidate_caches()
import torch
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (local verification)")
verification = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(verification.stdout, end="", flush=True)
verification.check_returncode()
from main_study.config import MainStudyConfig
from main_study.runner import run_main_study
from grid_study.config import GridTrancheConfig
from grid_study.runner import run_rarity_tranche, run_baseline_smoke, grid_source_fingerprint
if grid_source_fingerprint() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("Runtime source identity changed during setup.")


## Small baseline and three-cell smoke

Create a genuine tiny first-tranche baseline, exercise each scheduled cell with development seeds, and verify completed resume and parent preservation.

In [ ]:
BASELINE_SMOKE = RUNS / ("rarity-baseline-smoke-" + EXPECTED_SOURCE_SHA256[:12])
legacy_smoke_config = MainStudyConfig.from_file(PROJECT / "configs/main_study_smoke.json")
if legacy_smoke_config.mode != "smoke":
    raise RuntimeError("The baseline preflight must use smoke mode.")
legacy_smoke = run_baseline_smoke(legacy_smoke_config, BASELINE_SMOKE, device=DEVICE)
if legacy_smoke["status"] != "complete":
    raise RuntimeError("Baseline smoke incomplete. Rerun this third cell with the same folders.")
def _hash_files(folder):
    return {str(p.relative_to(folder)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(folder.rglob("*")) if p.is_file()}
parent_before = _hash_files(BASELINE_SMOKE)
SMOKE_RUN = RUNS / ("rarity-smoke-" + EXPECTED_SOURCE_SHA256[:12])
smoke_config = GridTrancheConfig.from_file(PROJECT / "configs/rarity_smoke.json")
if smoke_config.mode != "smoke":
    raise RuntimeError("The rarity preflight must use smoke mode.")
for _ in range(3):
    smoke = run_rarity_tranche(smoke_config, BASELINE_SMOKE, SMOKE_RUN, device=DEVICE)
    if smoke["status"] == "complete":
        break
if smoke["status"] != "complete":
    raise RuntimeError("Rarity smoke paused. Rerun this third cell with the same folders.")
before = _hash_files(SMOKE_RUN)
resumed = run_rarity_tranche(smoke_config, BASELINE_SMOKE, SMOKE_RUN, device=DEVICE)
if resumed != smoke or _hash_files(SMOKE_RUN) != before or _hash_files(BASELINE_SMOKE) != parent_before:
    raise RuntimeError("Completed resume changed the smoke results or read-only baseline.")
print("All three rarity cells, both strategies, saved baseline reference and immutable completed resume passed.")


## Run the next 18 populations

The production cells use the original frozen seed formula. All 100-trial populations retain the same training and evaluation recipe. The 2% baseline is inherited context and has separate compute accounting.

In [ ]:
RUN_NAME = "main-study-rarity-v1"
config = GridTrancheConfig.from_file(PROJECT / "configs" / ("rarity_tranche.json" if IN_COLAB else "rarity_smoke.json"))
if config.mode != ("production" if IN_COLAB else "smoke"):
    raise RuntimeError("The rarity run configuration has the wrong role.")
PARENT_RUN = RUNS / "main-study-first-tranche-v1" if IN_COLAB else BASELINE_SMOKE
print(json.dumps(config.to_dict(), indent=2))
report = run_rarity_tranche(config, PARENT_RUN, RUNS / RUN_NAME, device=DEVICE)
if not IN_COLAB:
    # The actual local cell exercises all three tiny cells across real invocations.
    for _ in range(2):
        if report["status"] == "complete":
            break
        report = run_rarity_tranche(config, PARENT_RUN, RUNS / RUN_NAME, device=DEVICE)
if report["status"] != "complete":
    print(report.get("message", "Rarity tranche paused with saved checkpoints."))
    print("Completed cells:", report["completed_cells"])
    print("Pending cells:", report["pending_cells"])
    print("Live runtime: rerun ONLY this fourth code cell with the same RUN_NAME.")
    print("After a deleted runtime: restore cells 1–3, then rerun cell 4.")


## Rarity comparison

Positive prediction gain means better distilled prediction. Negative assay error change means lower distilled behavioral error. Report the primary pair alongside diagnostic scores, probe fidelity, both checkpoint rules and all controls. Each rarity/strategy cell has three distinct populations. Ranges and directions are descriptive; rarity differences use independent populations.

In [ ]:
if report["status"] == "complete":
    from IPython.display import display, Markdown
    def _score(a, signed=False):
        if a["mean"] is None:
            return f"unavailable ({a['supported_seed_count']}/{a['planned_seed_count']})"
        value = f"{a['mean']:+.5f}" if signed else f"{a['mean']:.5f}"
        return value + f" [{a['min']:.5f}, {a['max']:.5f}] ({a['supported_seed_count']}/{a['planned_seed_count']})"
    cells = [("Verified baseline", report["baseline_reference"]["summary"])] + [(name, cell) for name, cell in report["cells"].items()]
    rows = ["**Rarity comparison at a fixed history length**", "",
            "Within each population, distilled is paired with solo. Different rarity levels use distinct populations.", "",
            "| Rare fraction | Strategy | Prediction gain | Assay error change | Solo diagnostic KL | Distilled diagnostic KL |",
            "| --- | --- | ---: | ---: | ---: | ---: |"]
    for label, cell in cells:
        fraction = cell["config"]["rare_fraction"]
        for kind_name, kind in cell["kinds"].items():
            group = kind["groups"]["rare"]
            primary = group["primary"]["population"]
            rows.append("| " + " | ".join([f"{100*fraction:g}%", kind_name,
                _score(primary["prediction_gain"], True), _score(primary["behavioral_distortion"], True),
                _score(group["models"]["solo_population"]["diagnostic_kl"]),
                _score(group["models"]["distilled_population"]["diagnostic_kl"])]) + " |")
    display(Markdown("\n".join(rows)))
    for label, cell in cells:
        rows = [f"**{label}: {100*cell['config']['rare_fraction']:g}% rare, {cell['config']['n_trials']} trials**", ""]
        for kind_name, kind in cell["kinds"].items():
            for group_name, group in kind["groups"].items():
                rows += [f"**{kind_name} / {group_name}**", "",
                         f"Diagnostic trials / people: {_score(group['diagnostic_count'])} / {_score(group['diagnostic_subject_count'])}", "",
                         "| Model | Test NLL | True-policy KL | Diagnostic KL | Assay RMS | Probe KL |",
                         "| --- | ---: | ---: | ---: | ---: | ---: |"]
                for model, scores in group["models"].items():
                    rows.append("| " + " | ".join([model]+[_score(scores[key]) for key in ("nll", "kl", "diagnostic_kl", "signature_rms_error", "probe_kl")]) + " |")
                for mode in ("population", "individual"):
                    primary = group["primary"][mode]
                    rows += ["", f"{mode} checkpoint rule: prediction gain {_score(primary['prediction_gain'], True)}; assay error change {_score(primary['behavioral_distortion'], True)}; positive-both fraction {_score(primary['quadrant_fraction'])}."]
            for population in kind["per_seed"]:
                print(label, kind_name, population["seed"], json.dumps({group:v["primary"] for group,v in population["groups"].items()}))
        display(Markdown("\n".join(rows)))
    print(json.dumps(report["timing"], indent=2))
    print("Keep the full main-study-rarity-v1 and", SMOKE_RUN.name, "folders for review.")
    print("Baseline training is inherited context. New training, setup, evaluation and idle billing have separate accounting.")
    print("Disconnect the GPU when complete.")
else:
    print("The rarity tranche is incomplete. Rerun only cell 4 in the live runtime.")
    print("All18 planned populations must finish before the final rarity tables are displayed.")
